# 🎨 Prompt Engineering Patterns

**Master the art and science of communicating with LLMs**

Learn proven prompt patterns that professional AI engineers use to get consistent, high-quality results from language models.

---

## 📋 Overview

**What you'll learn:**
- Zero-shot, one-shot, and few-shot prompting
- Chain-of-thought reasoning
- System vs user messages
- Prompt templates and reusability
- Testing and optimizing prompts

**Prerequisites:** 
- Completed `02_llm_basics/01_first_api_call.ipynb`
- Understanding of API calls
- At least one API key configured

**Time estimate:** ⏱️ 60-90 minutes

**Difficulty:** 🟡 Intermediate

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. ✅ Write effective prompts using proven patterns
2. ✅ Use examples to guide model behavior (few-shot learning)
3. ✅ Implement chain-of-thought reasoning for complex tasks
4. ✅ Create reusable prompt templates
5. ✅ Test and optimize prompts systematically

---

## 📖 What is Prompt Engineering?

### The Analogy

Think of an LLM like a highly capable assistant who:
- Knows a lot but needs clear instructions
- Can do many things but needs examples
- Works best with structured tasks

**Bad instruction:** "Do the thing."
**Good instruction:** "Summarize this article in 3 bullet points, focusing on key findings."

### Why It Matters

The **same model** with **different prompts** can produce:
```
Bad prompt → Vague, wrong, or useless output
Good prompt → Accurate, useful, consistent output
```

**Real-world impact:**
- Customer support bot: 60% → 95% accuracy
- Code generation: Buggy → Production-ready
- Content creation: Generic → On-brand

### The 4 Principles of Good Prompts

1. **Clear** - No ambiguity about what you want
2. **Specific** - Provide context and constraints
3. **Examples** - Show, don't just tell
4. **Structure** - Format the output you want

---

In [ ]:
# Setup
import os
import time
from dotenv import load_dotenv
from typing import List, Dict, Any
import json

load_dotenv()

# Import our helpers from previous notebook
from groq import Groq

# We'll use Groq for speed and cost-effectiveness
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

def call_llm(prompt: str, system_prompt: str = None, temperature: float = 0.7, **kwargs) -> str:
    """
    Simple wrapper for LLM calls.
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=messages,
        temperature=temperature,
        **kwargs
    )
    
    return response.choices[0].message.content

print("✅ Setup complete! Ready to learn prompt engineering.")

---

## 🎯 Pattern 1: Zero-Shot Prompting

**What it is:** Ask the model to do something without examples.

**When to use:** 
- Simple, straightforward tasks
- General knowledge questions
- When examples aren't available

**Structure:**
```
[Task description] + [Input] + [Output format]
```

In [ ]:
# Example 1: Bad zero-shot prompt
print("❌ Bad Zero-Shot Prompt:")
print("-" * 80)

bad_prompt = "Sentiment of: 'This product is okay'"
print(f"Prompt: {bad_prompt}\n")

response = call_llm(bad_prompt, max_tokens=50)
print(f"Response: {response}")
print("\n⚠️ Problem: Vague, verbose, inconsistent format\n")

print("=" * 80)

# Example 2: Good zero-shot prompt
print("\n✅ Good Zero-Shot Prompt:")
print("-" * 80)

good_prompt = """Classify the sentiment of the following text as exactly one of: positive, negative, or neutral.

Text: "This product is okay"

Sentiment:"""

print(f"Prompt:\n{good_prompt}\n")

response = call_llm(good_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {response}")
print("\n✅ Clear, concise, consistent format")

### 💡 Zero-Shot Best Practices

**DO:**
- ✅ Be specific about the task
- ✅ Define the output format
- ✅ Set constraints (length, style, etc.)
- ✅ Use temperature=0 for consistency

**DON'T:**
- ❌ Assume the model knows your intent
- ❌ Leave output format ambiguous
- ❌ Use vague language
- ❌ Forget to test edge cases

---

## 🎯 Pattern 2: Few-Shot Prompting

**What it is:** Provide examples of the task before asking the model to perform it.

**When to use:**
- Complex tasks
- Specific formatting requirements
- Domain-specific knowledge
- Inconsistent zero-shot results

**Structure:**
```
[Task description]
[Example 1]
[Example 2]
[Example 3]
[New input]
```

**Rule of thumb:** 3-5 examples is usually optimal

In [ ]:
# Comparison: Zero-shot vs Few-shot
print("🔬 Comparing Zero-Shot vs Few-Shot\n")
print("=" * 80)

# Task: Extract product name and price from text
test_input = "Looking at the MacBook Pro 16-inch for $2,399 - seems expensive!"

# Zero-shot attempt
print("Zero-Shot Approach:")
print("-" * 80)

zero_shot = f"""Extract the product name and price from this text:

{test_input}

Output format: Product: [name], Price: [price]"""

response_zero = call_llm(zero_shot, temperature=0.0, max_tokens=50)
print(f"Response: {response_zero}\n")

# Few-shot approach
print("=" * 80)
print("\nFew-Shot Approach:")
print("-" * 80)

few_shot = """Extract the product name and price from customer messages. Return ONLY in this format:
Product: [name], Price: [price]

Examples:

Text: "I want to buy the iPhone 15 Pro which costs $999"
Product: iPhone 15 Pro, Price: $999

Text: "The Tesla Model 3 is listed at $40,000 on the website"
Product: Tesla Model 3, Price: $40,000

Text: "Thinking about getting AirPods Max, they're $549"
Product: AirPods Max, Price: $549

Now extract from:
Text: "{}" 
""".format(test_input)

response_few = call_llm(few_shot, temperature=0.0, max_tokens=50)
print(f"Response: {response_few}")

print("\n💡 Few-shot is more reliable and follows the format exactly!")

In [ ]:
# Advanced Few-Shot: Teaching Custom Format
print("🎨 Teaching Custom JSON Format with Few-Shot\n")
print("=" * 80)

few_shot_json = """Convert customer support messages to structured JSON.

Examples:

Input: "I'm having trouble logging in, getting error 500"
Output: {"category": "technical", "priority": "high", "keywords": ["login", "error 500"]}

Input: "When will my order #12345 arrive?"
Output: {"category": "shipping", "priority": "medium", "keywords": ["order", "#12345", "delivery"]}

Input: "Just wanted to say your service is great!"
Output: {"category": "feedback", "priority": "low", "keywords": ["positive", "service"]}

Now convert:
Input: "The app keeps crashing when I try to checkout"
Output:"""

response = call_llm(few_shot_json, temperature=0.0, max_tokens=100)
print(f"Response:\n{response}")

# Validate it's JSON
try:
    parsed = json.loads(response)
    print("\n✅ Valid JSON!")
    print(f"Category: {parsed.get('category')}")
    print(f"Priority: {parsed.get('priority')}")
    print(f"Keywords: {', '.join(parsed.get('keywords', []))}")
except:
    print("\n⚠️ Not valid JSON, but close!")

### 💡 Few-Shot Best Practices

**Choosing Examples:**
1. **Diverse** - Cover different scenarios
2. **Representative** - Match real use cases
3. **Clear** - Unambiguous input/output
4. **Consistent** - Same format across examples

**How many examples?**
```
1 example  = One-shot (for simple tasks)
3 examples = Few-shot (most common)
5+ examples = Many-shot (complex tasks)
```

**Cost consideration:**
- More examples = more input tokens = higher cost
- But: Better accuracy often worth it!
- Optimize by finding minimum effective examples

---

## 🎯 Pattern 3: Chain-of-Thought (CoT)

**What it is:** Ask the model to "think step-by-step" before answering.

**When to use:**
- Math problems
- Complex reasoning
- Multi-step tasks
- When you need to verify logic

**Magic phrase:** "Let's think step by step."

**Why it works:** Forces the model to show its reasoning, leading to more accurate results.

In [ ]:
# Comparison: Direct vs Chain-of-Thought
print("🧠 Chain-of-Thought Demonstration\n")
print("=" * 80)

problem = """A store has 23 apples. They sell 14 apples in the morning and 7 apples in the afternoon. 
They receive a delivery of 18 apples in the evening. How many apples do they have now?"""

# Without CoT
print("Without Chain-of-Thought:")
print("-" * 80)

direct_prompt = f"{problem}\n\nAnswer:"
response_direct = call_llm(direct_prompt, temperature=0.0, max_tokens=50)
print(f"Response: {response_direct}\n")

# With CoT
print("=" * 80)
print("\nWith Chain-of-Thought:")
print("-" * 80)

cot_prompt = f"{problem}\n\nLet's solve this step by step:\n"
response_cot = call_llm(cot_prompt, temperature=0.0, max_tokens=200)
print(f"Response:\n{response_cot}")

print("\n💡 CoT shows reasoning, making it easier to verify and debug!")

In [ ]:
# Few-Shot + Chain-of-Thought (Most Powerful!)
print("🚀 Combining Few-Shot + Chain-of-Thought\n")
print("=" * 80)

few_shot_cot = """Solve these word problems step by step.

Example 1:
Problem: John has 15 cookies. He eats 3 and gives 5 to his friend. How many does he have left?
Solution:
Step 1: John starts with 15 cookies
Step 2: He eats 3: 15 - 3 = 12
Step 3: He gives away 5: 12 - 5 = 7
Answer: 7 cookies

Example 2:
Problem: A parking lot has 50 spaces. 32 are occupied. 8 cars leave and 12 cars arrive. How many spaces are occupied?
Solution:
Step 1: Start with 32 occupied spaces
Step 2: 8 cars leave: 32 - 8 = 24
Step 3: 12 cars arrive: 24 + 12 = 36
Answer: 36 spaces occupied

Now solve:
Problem: A library has 200 books. They lend out 45 books on Monday and 38 books on Tuesday. 
They receive 60 books from donations. How many books does the library have now?
Solution:"""

response = call_llm(few_shot_cot, temperature=0.0, max_tokens=250)
print(f"Response:\n{response}")

print("\n✅ This combines the best of both: examples + step-by-step reasoning!")

### 💡 Chain-of-Thought Best Practices

**When to use CoT:**
- ✅ Math and logic problems
- ✅ Complex reasoning tasks
- ✅ When accuracy is critical
- ✅ When you need to verify reasoning

**When NOT to use CoT:**
- ❌ Simple tasks (overkill, slower, costs more)
- ❌ Creative writing (too structured)
- ❌ When you only need the final answer quickly

**CoT Variations:**
```python
"Let's think step by step"           # Most common
"Let's solve this systematically"    # Alternative
"Let's break this down"              # Simpler version
"First... Then... Finally..."        # Structured version
```

---

## 🎯 Pattern 4: System Prompts

**What it is:** Set the model's role and behavior before the conversation.

**Structure:**
```python
system_prompt = "You are a [role] that [behavior]."
user_prompt = "[actual task]"
```

**Why it matters:**
- Sets consistent tone and style
- Defines constraints
- Persists across conversations

In [ ]:
# Demonstration: Different system prompts, same query
print("🎭 System Prompt Comparison\n")
print("=" * 80)

user_query = "Explain what a neural network is."

# System Prompt 1: Expert mode
print("System Prompt 1: Technical Expert")
print("-" * 80)

system1 = """You are a senior machine learning engineer. 
Explain concepts using technical terminology and assume the user has a CS background."""

response1 = call_llm(user_query, system_prompt=system1, temperature=0.7, max_tokens=150)
print(f"\nResponse:\n{response1}\n")

# System Prompt 2: Beginner mode
print("=" * 80)
print("\nSystem Prompt 2: Friendly Teacher")
print("-" * 80)

system2 = """You are a friendly teacher explaining to a 10-year-old. 
Use simple language, analogies, and avoid jargon."""

response2 = call_llm(user_query, system_prompt=system2, temperature=0.7, max_tokens=150)
print(f"\nResponse:\n{response2}\n")

# System Prompt 3: Business mode
print("=" * 80)
print("\nSystem Prompt 3: Business Consultant")
print("-" * 80)

system3 = """You are a business consultant. 
Explain technical concepts in terms of business value and ROI."""

response3 = call_llm(user_query, system_prompt=system3, temperature=0.7, max_tokens=150)
print(f"\nResponse:\n{response3}")

print("\n💡 Same question, three completely different styles!")

### 📋 System Prompt Templates

Here are production-ready templates:

In [ ]:
# Production System Prompt Templates
SYSTEM_PROMPTS = {
    'customer_support': """
You are a helpful customer support agent for [Company Name].
- Be polite, professional, and empathetic
- If you don't know something, admit it and offer to escalate
- Keep responses under 100 words unless more detail is needed
- Never make promises about refunds or policies without checking
""",
    
    'code_reviewer': """
You are an expert code reviewer with 10+ years of experience.
- Focus on: security, performance, readability, best practices
- Provide specific suggestions, not just criticisms
- Rate severity: Critical, High, Medium, Low
- Keep feedback constructive and actionable
""",
    
    'data_analyst': """
You are a senior data analyst.
- Provide data-driven insights
- Always question data quality and assumptions
- Suggest visualizations when appropriate
- Explain statistical concepts clearly
""",
    
    'content_writer': """
You are a professional content writer.
- Write in [Brand Voice: friendly/professional/casual]
- Target audience: [Define audience]
- Tone: [conversational/formal/technical]
- Always include a clear call-to-action
- Use active voice and keep sentences under 20 words
""",
    
    'json_api': """
You are a JSON API. Always respond with valid JSON only.
- No explanatory text before or after JSON
- Use double quotes for strings
- Include error field if something goes wrong
Example: {"status": "success", "data": {}, "error": null}
"""
}

# Example: JSON API system prompt
print("🤖 JSON API Example\n")
print("=" * 80)

response = call_llm(
    "Extract the name and email from: 'Contact John Doe at john@example.com'",
    system_prompt=SYSTEM_PROMPTS['json_api'],
    temperature=0.0,
    max_tokens=100
)

print(f"Response:\n{response}")

try:
    data = json.loads(response)
    print("\n✅ Valid JSON!")
    print(json.dumps(data, indent=2))
except:
    print("\n⚠️ Not valid JSON")

---

## 🎯 Pattern 5: Prompt Templates

**What it is:** Reusable prompt structures with placeholders.

**Why use them:**
- Consistency across team
- Easy to test and iterate
- Maintainable and versioned
- Scalable to many use cases

In [ ]:
class PromptTemplate:
    """
    Reusable prompt template with variable substitution.
    """
    
    def __init__(self, template: str, name: str = None):
        self.template = template
        self.name = name or "Unnamed Template"
    
    def format(self, **kwargs) -> str:
        """Fill in template variables."""
        return self.template.format(**kwargs)
    
    def __repr__(self):
        return f"PromptTemplate(name='{self.name}')"

# Create templates for common tasks
TEMPLATES = {
    'sentiment': PromptTemplate(
        name="Sentiment Analysis",
        template="""Classify the sentiment of the following text as one of: {sentiments}

Text: "{text}"

Sentiment (one word only):"""
    ),
    
    'summarize': PromptTemplate(
        name="Summarization",
        template="""Summarize the following text in {num_sentences} sentences.
Focus on {focus}.

Text: {text}

Summary:"""
    ),
    
    'translate': PromptTemplate(
        name="Translation",
        template="""Translate the following text from {source_lang} to {target_lang}.
Maintain {style} tone.

Text: "{text}"

Translation:"""
    ),
    
    'extract': PromptTemplate(
        name="Information Extraction",
        template="""Extract {fields} from the following text.
Return as JSON with keys: {json_keys}

Text: "{text}"

JSON:"""
    ),
}

# Test the templates
print("📝 Prompt Template Examples\n")
print("=" * 80)

# Example 1: Sentiment
print("Template: Sentiment Analysis")
print("-" * 80)

sentiment_prompt = TEMPLATES['sentiment'].format(
    sentiments="positive, negative, neutral",
    text="I love this product, but the shipping was slow."
)
print(f"Generated Prompt:\n{sentiment_prompt}\n")

response = call_llm(sentiment_prompt, temperature=0.0, max_tokens=10)
print(f"Response: {response}\n")

# Example 2: Summarization
print("=" * 80)
print("\nTemplate: Summarization")
print("-" * 80)

article = """Artificial intelligence has made remarkable progress in recent years. 
Machine learning models can now perform tasks that were once thought impossible, 
from generating human-like text to recognizing images with superhuman accuracy. 
However, these advances also raise important questions about ethics, bias, and 
the future of work. As AI continues to evolve, society must grapple with how to 
harness its benefits while mitigating potential harms."""

summary_prompt = TEMPLATES['summarize'].format(
    num_sentences=2,
    focus="key challenges and opportunities",
    text=article
)

response = call_llm(summary_prompt, temperature=0.3, max_tokens=100)
print(f"Response:\n{response}")

print("\n💡 Templates ensure consistency and make prompt engineering scalable!")

---

## 🧪 Testing and Optimizing Prompts

**The Process:**
1. Start with a baseline prompt
2. Create a test set (5-10 examples)
3. Iterate and measure improvements
4. A/B test in production

In [ ]:
def test_prompt(prompt_template: str, test_cases: List[Dict], **kwargs) -> Dict:
    """
    Test a prompt on multiple inputs and measure success.
    
    Args:
        prompt_template: Prompt with {input} placeholder
        test_cases: List of {"input": ..., "expected": ...}
    
    Returns:
        Results dictionary with success rate
    """
    results = []
    
    for i, case in enumerate(test_cases, 1):
        prompt = prompt_template.format(input=case['input'])
        response = call_llm(prompt, **kwargs)
        
        # Simple success check (contains expected answer)
        success = case['expected'].lower() in response.lower()
        
        results.append({
            'test': i,
            'input': case['input'],
            'expected': case['expected'],
            'response': response,
            'success': success
        })
    
    success_rate = sum(r['success'] for r in results) / len(results)
    
    return {
        'results': results,
        'success_rate': success_rate,
        'total_tests': len(results)
    }

# Example: Testing sentiment analysis prompts
print("🧪 Prompt Testing Example\n")
print("=" * 80)

test_cases = [
    {"input": "This is the best thing ever!", "expected": "positive"},
    {"input": "I hate this product.", "expected": "negative"},
    {"input": "It's okay, nothing special.", "expected": "neutral"},
    {"input": "Absolutely terrible experience.", "expected": "negative"},
    {"input": "Love it! Highly recommend.", "expected": "positive"},
]

# Test Prompt Version 1 (Basic)
print("Testing Prompt V1 (Basic):")
print("-" * 80)

prompt_v1 = "What is the sentiment: {input}"

results_v1 = test_prompt(prompt_v1, test_cases, temperature=0.0, max_tokens=10)

print(f"Success Rate: {results_v1['success_rate']:.1%}\n")

# Test Prompt Version 2 (Improved)
print("=" * 80)
print("\nTesting Prompt V2 (Improved):")
print("-" * 80)

prompt_v2 = """Classify the sentiment as one word: positive, negative, or neutral.

Text: "{input}"

Sentiment:"""

results_v2 = test_prompt(prompt_v2, test_cases, temperature=0.0, max_tokens=10)

print(f"Success Rate: {results_v2['success_rate']:.1%}\n")

# Show detailed results
print("=" * 80)
print("\nDetailed Results (V2):")
for r in results_v2['results']:
    status = "✅" if r['success'] else "❌"
    print(f"{status} Test {r['test']}: {r['input'][:40]}...")
    print(f"   Expected: {r['expected']} | Got: {r['response'][:20]}")

print("\n💡 Always test your prompts on real data before production!")

---

## 🎯 Exercise: Build Your Own Prompt

### Challenge: Email Classifier

**Task:** Create a prompt that classifies emails into categories.

**Requirements:**
1. Categories: sales, support, billing, other
2. Include urgency level: low, medium, high
3. Extract key action items
4. Return as JSON
5. Test on 5 example emails

**Starter code:**

In [ ]:
def classify_email(email_text: str) -> Dict:
    """
    Classify email using your custom prompt.
    
    Returns JSON with: category, urgency, action_items
    """
    # TODO: Create your prompt here
    prompt = f"""YOUR PROMPT HERE
    
Email: {email_text}
    """
    
    response = call_llm(prompt, temperature=0.0, max_tokens=150)
    
    # Parse JSON response
    try:
        return json.loads(response)
    except:
        return {"error": "Invalid JSON response"}

# Test emails
test_emails = [
    "Hi, I'm interested in your enterprise plan. Can we schedule a demo this week?",
    "URGENT: My account has been locked for 2 hours. I need immediate help!",
    "Question about my last invoice - I was charged twice for the same service.",
    "Just wanted to say your product is great! Keep up the good work.",
    "Our credit card on file is about to expire. Please update billing information.",
]

# Test your classifier
for i, email in enumerate(test_emails, 1):
    result = classify_email(email)
    print(f"\nTest {i}: {email[:50]}...")
    print(f"Result: {json.dumps(result, indent=2)}")

### 💡 Solution

In [ ]:
# Solution implementation
def classify_email_solution(email_text: str) -> Dict:
    """
    Classify email using a well-designed prompt.
    """
    prompt = f"""Classify the following email and extract key information.

Categories: sales, support, billing, other
Urgency levels: low, medium, high

Return ONLY valid JSON with this structure:
{{
  "category": "[one of: sales, support, billing, other]",
  "urgency": "[one of: low, medium, high]",
  "action_items": ["list", "of", "actions"],
  "summary": "brief one-sentence summary"
}}

Email: "{email_text}"

JSON:"""
    
    response = call_llm(prompt, temperature=0.0, max_tokens=150)
    
    try:
        return json.loads(response)
    except:
        return {"error": "Invalid JSON", "raw_response": response}

# Test the solution
print("✅ Solution Example:\n")
print("=" * 80)

test_email = "URGENT: My account has been locked for 2 hours. I need immediate help!"
result = classify_email_solution(test_email)

print(f"Email: {test_email}\n")
print(f"Classification:\n{json.dumps(result, indent=2)}")

---

## ⚠️ Common Pitfalls

### 1. Vague Instructions
```python
# ❌ BAD
"Analyze this text"

# ✅ GOOD
"Analyze this customer review for sentiment (positive/negative/neutral) and key themes"
```

### 2. No Output Format
```python
# ❌ BAD
"Extract the date from this text"

# ✅ GOOD
"Extract the date from this text. Return ONLY in YYYY-MM-DD format."
```

### 3. Too Many Instructions
```python
# ❌ BAD (overwhelming)
"Summarize, translate, analyze sentiment, extract keywords, rate quality, suggest improvements..."

# ✅ GOOD (focused)
"Summarize this article in 3 bullet points."
```

### 4. Not Testing Edge Cases
```python
# Always test:
- Empty input
- Very long input
- Ambiguous input
- Invalid formats
- Unexpected languages
```

### 5. Ignoring Cost
```python
# Few-shot with 10 examples might be overkill
# Test if 3 examples work just as well
# Fewer examples = lower cost per call
```

---

## 🏭 Production Best Practices

### 1. Version Control Prompts
```python
# Store prompts in files, not code
prompts/
  v1_sentiment.txt
  v2_sentiment.txt
  v3_sentiment.txt
```

### 2. A/B Test Prompts
```python
# Run both versions, measure performance
if random.random() < 0.5:
    response = call_with_prompt_v1()
else:
    response = call_with_prompt_v2()

log_metrics(prompt_version, accuracy, latency)
```

### 3. Monitor Prompt Performance
```python
# Track metrics:
- Success rate
- User feedback
- Token usage
- Latency
- Cost per request
```

### 4. Create Prompt Library
```python
# Build reusable templates
# Document what works
# Share across team
```

### 5. Implement Fallbacks
```python
# If primary prompt fails, try simpler version
try:
    result = complex_prompt()
except:
    result = simple_prompt()
```

---

## 📚 Further Reading

### Research Papers
- [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)
- [Few-Shot Learning with Language Models](https://arxiv.org/abs/2005.14165)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)

### Tools & Resources
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Prompt Library](https://docs.anthropic.com/claude/prompt-library)
- [LangChain Prompt Templates](https://python.langchain.com/docs/modules/model_io/prompts/)

### Next Notebooks
- `03_prompt_engineering/02_system_prompts.ipynb` - Deep dive into system messages
- `03_prompt_engineering/03_structured_output.ipynb` - JSON mode and Pydantic
- `03_prompt_engineering/05_chain_of_thought.ipynb` - Advanced CoT techniques

---

## ✅ Summary

### What You Learned

1. ✅ **Prompt Patterns**
   - Zero-shot: Direct instructions
   - Few-shot: Learning from examples
   - Chain-of-thought: Step-by-step reasoning
   - System prompts: Setting context
   - Templates: Reusable structures

2. ✅ **Best Practices**
   - Be specific and clear
   - Provide examples
   - Define output format
   - Test systematically
   - Version and iterate

3. ✅ **Testing & Optimization**
   - Create test sets
   - Measure success rates
   - A/B test variations
   - Monitor in production

### Key Takeaways

💡 **Good prompts = Better results** - Worth the investment!

📝 **Few-shot beats zero-shot** - For complex/specific tasks

🧠 **Chain-of-thought** - For reasoning and math

🎭 **System prompts** - For consistent behavior

🧪 **Always test** - Don't assume, measure!

### Prompt Engineering Checklist

Before deploying any prompt:
- [ ] Clear task description
- [ ] Explicit output format
- [ ] Appropriate examples (if needed)
- [ ] System prompt (if needed)
- [ ] Tested on 5-10 cases
- [ ] Edge cases covered
- [ ] Cost estimated
- [ ] Version controlled

### Next Steps

Continue your prompt engineering journey:

1. **Practice:** Try each pattern on your own use cases
2. **Build:** Create a prompt library for your projects
3. **Test:** Set up systematic testing
4. **Next notebook:** `03_prompt_engineering/02_system_prompts.ipynb`

---

## 🎉 Congratulations!

You now know the core patterns of prompt engineering!

**You can:**
- Design effective prompts
- Choose the right pattern for each task
- Test and optimize systematically
- Build production-ready prompts

**This skill alone can dramatically improve your AI applications!** 🚀

---

*Keep experimenting - prompt engineering is part art, part science!*